In [0]:
df = spark.table("emails_spam_detection")
df.show(5)

+--------------------+-----+
|                text|label|
+--------------------+-----+
|Subject: naturall...|    1|
|Subject: the stoc...|    1|
|Subject: unbeliev...|    1|
|Subject: 4 color ...|    1|
|Subject: do not h...|    1|
+--------------------+-----+
only showing top 5 rows


In [0]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# Step 1: Tokenize (split text into words)
tokenizer = Tokenizer(inputCol="text", outputCol="words")
wordsData = tokenizer.transform(df)

# Step 2: Remove common words (like "the", "is")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
filteredData = remover.transform(wordsData)

filteredData.select("filtered").show(5)

+--------------------+
|            filtered|
+--------------------+
|[subject:, natura...|
|[subject:, stock,...|
|[subject:, unbeli...|
|[subject:, 4, col...|
|[subject:, money,...|
+--------------------+
only showing top 5 rows


In [0]:
from pyspark.ml.feature import HashingTF, IDF

# Convert words to numeric features
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures")
featurizedData = hashingTF.transform(filteredData)

# Apply TF-IDF
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)

In [0]:
from pyspark.sql.functions import col

finalData = rescaledData.withColumn("label", col("label").cast("double"))

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")

model = lr.fit(finalData)

predictions = model.transform(finalData)

predictions.select("text", "prediction").show(5)

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-4656458394279786>, line 5
      1 from pyspark.ml.classification import LogisticRegression
      3 lr = LogisticRegression(featuresCol="features", labelCol="label")
----> 5 model = lr.fit(finalData)
      7 predictions = model.transform(finalData)
      9 predictions.select("text", "prediction").show(5)

File /databricks/python_shell/lib/dbruntime/MLWorkloadsInstrumentation/_pyspark.py:30, in _create_patch_function.<locals>.patched_method(self, *args, **kwargs)
     28 call_succeeded = False
     29 try:
---> 30     result = original_method(self, *args, **kwargs)
     31     call_succeeded = True
     32     return result

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/base.py:203, in Estimator.fit(self, dataset, params)
    201         return self.copy(params)._fit(dataset)
    202     else:
--> 203      

In [0]:
df.select("label").distinct().show()

+--------------------+
|               label|
+--------------------+
|                   1|
|                   0|
| its termination ...|
|   mr suresh prabhu |
|                NULL|
+--------------------+



In [0]:
from pyspark.sql.functions import when, col

finalData = rescaledData.withColumn(
    "label_num",
    when(col("label") == "spam", 1)
    .when(col("label") == "ham", 0)
    .otherwise(col("label").cast("double"))
)

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label_num")

model = lr.fit(finalData)

predictions = model.transform(finalData)

predictions.select("text", "prediction").show(5)

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-4656458394279789>, line 5
      1 from pyspark.ml.classification import LogisticRegression
      3 lr = LogisticRegression(featuresCol="features", labelCol="label_num")
----> 5 model = lr.fit(finalData)
      7 predictions = model.transform(finalData)
      9 predictions.select("text", "prediction").show(5)

File /databricks/python_shell/lib/dbruntime/MLWorkloadsInstrumentation/_pyspark.py:30, in _create_patch_function.<locals>.patched_method(self, *args, **kwargs)
     28 call_succeeded = False
     29 try:
---> 30     result = original_method(self, *args, **kwargs)
     31     call_succeeded = True
     32     return result

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/base.py:203, in Estimator.fit(self, dataset, params)
    201         return self.copy(params)._fit(dataset)
    202     else:
--> 203  

In [0]:
df.select("label").distinct().show(20, False)

+--------------------------------------------------------------------------------------------+
|label                                                                                       |
+--------------------------------------------------------------------------------------------+
|1                                                                                           |
|0                                                                                           |
| its termination would not  have such a phenomenal impact on the power situation .  however |
| mr suresh prabhu                                                                           |
|NULL                                                                                        |
+--------------------------------------------------------------------------------------------+



In [0]:
from pyspark.sql.functions import col

# Keep only valid rows where label is 0 or 1
cleanFinal = rescaledData.filter(
    (col("label") == 0) | (col("label") == 1) |
    (col("label") == "0") | (col("label") == "1")
)

# Convert safely
cleanFinal = cleanFinal.withColumn("label_num", col("label").cast("double"))

In [0]:
df.show(5)
df.printSchema()

+--------------------+-----+
|                text|label|
+--------------------+-----+
|Subject: naturall...|    1|
|Subject: the stoc...|    1|
|Subject: unbeliev...|    1|
|Subject: 4 color ...|    1|
|Subject: do not h...|    1|
+--------------------+-----+
only showing top 5 rows
root
 |-- text: string (nullable = true)
 |-- label: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

# Rename correctly (swap if needed)
fixedDF = df.select(
    col("text").alias("email_text"),
    col("label").alias("spam_label")
)

In [0]:
from pyspark.sql.functions import when

fixedDF = fixedDF.withColumn(
    "label_num",
    when(col("spam_label") == "spam", 1)
    .when(col("spam_label") == "ham", 0)
    .otherwise(col("spam_label").cast("double"))
)

In [0]:
df = spark.table("emails_spam_detection")
df.show(5)

+--------------------+-----+
|                text|label|
+--------------------+-----+
|Subject: naturall...|    1|
|Subject: the stoc...|    1|
|Subject: unbeliev...|    1|
|Subject: 4 color ...|    1|
|Subject: do not h...|    1|
+--------------------+-----+
only showing top 5 rows


+--------------------+-----+
|                text|label|
+--------------------+-----+
|Subject: naturall...|    1|
|Subject: the stoc...|    1|
|Subject: unbeliev...|    1|
|Subject: 4 color ...|    1|
|Subject: do not h...|    1|
+--------------------+-----+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, when, trim

cleanData = rescaledData.withColumn(
    "label_clean",
    trim(col("label"))
)

# Keep only valid labels (0 or 1)
cleanData = cleanData.filter(
    (col("label_clean") == "0") | (col("label_clean") == "1")
)

# Convert safely
cleanData = cleanData.withColumn(
    "label_num",
    col("label_clean").cast("double")
)

In [0]:
cleanData.select("label_clean", "label_num").show(10, False)

+-----------+---------+
|label_clean|label_num|
+-----------+---------+
|1          |1.0      |
|1          |1.0      |
|1          |1.0      |
|1          |1.0      |
|1          |1.0      |
|1          |1.0      |
|1          |1.0      |
|1          |1.0      |
|1          |1.0      |
|1          |1.0      |
+-----------+---------+
only showing top 10 rows


In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label_num")

model = lr.fit(cleanData)

predictions = model.transform(cleanData)

predictions.select("text", "prediction").show(5)

+--------------------+----------+
|                text|prediction|
+--------------------+----------+
|Subject: naturall...|       1.0|
|Subject: the stoc...|       1.0|
|Subject: unbeliev...|       1.0|
|Subject: 4 color ...|       1.0|
|Subject: do not h...|       1.0|
+--------------------+----------+
only showing top 5 rows


In [0]:
df.groupBy("label").count().show()

df.selectExpr("length(text) as email_length").show()

df.describe().show()

+--------------------+-----+
|               label|count|
+--------------------+-----+
|                   1| 1368|
|                   0| 4358|
|                NULL|    2|
| its termination ...|    1|
|   mr suresh prabhu |    1|
+--------------------+-----+

+------------+
|email_length|
+------------+
|        1484|
|         598|
|         448|
|         500|
|         235|
|         478|
|        9340|
|         446|
|         507|
|         446|
|         709|
|         446|
|         879|
|        8196|
|         684|
|        1000|
|         215|
|         663|
|          80|
|        2560|
+------------+
only showing top 20 rows
+-------+--------------------+--------------------+
|summary|                text|               label|
+-------+--------------------+--------------------+
|  count|                5730|                5728|
|   mean|                NULL| 0.23891023402025846|
| stddev|                NULL|  0.4264550330012776|
|    min|Subject:   http :...| its termin